# Sistema de Previsão de Demanda — LH Nautical

## Produto: Bússola de Bordo 702

Este notebook implementa o baseline solicitado no desafio técnico utilizando os arquivos:

- `products.csv`
- `product_variants.csv`
- `orders.csv`
- `order_items.csv`

### Premissas

- Treino: dados até **31/12/2025**.
- Teste: **01/01/2026 a 31/03/2026**.
- Granularidade: **mensal**.
- Produto: **Bússola de Bordo 702**.
- Baseline: **média móvel dos últimos 3 meses**.
- Métrica: **MAE — Mean Absolute Error**.
- Pedidos válidos para representar venda: `paid` e `confirmed`.

> O produto aparece mais de uma vez no cadastro. Por isso, todas as ocorrências de `product_id` com o nome exato são consideradas.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Configuração dos caminhos

In [2]:
# Caminho padrão dentro do container Docker.
# Caso esteja executando fora do Docker, altere DATA_DIR para o diretório local dos CSVs.

DATA_DIR = Path('/workspace/data/raw')

# Fallback útil para execução local a partir da raiz do projeto.
if not DATA_DIR.exists():
    DATA_DIR = Path('data/raw')

FILES = {
    'products': DATA_DIR / 'products.csv',
    'variants': DATA_DIR / 'product_variants.csv',
    'orders': DATA_DIR / 'orders.csv',
    'order_items': DATA_DIR / 'order_items.csv',
}

missing = [str(path) for path in FILES.values() if not path.exists()]

if missing:
    raise FileNotFoundError(
        'Arquivos não encontrados:\n' + '\n'.join(missing)
    )

print('Diretório de dados:', DATA_DIR.resolve())

Diretório de dados: /workspace/data/raw


## 2. Leitura dos datasets

In [3]:
products = pd.read_csv(FILES['products'])
variants = pd.read_csv(FILES['variants'])
orders = pd.read_csv(FILES['orders'])
order_items = pd.read_csv(FILES['order_items'])

for name, df in {
    'products': products,
    'product_variants': variants,
    'orders': orders,
    'order_items': order_items,
}.items():
    print(f'{name}: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

products: 500 linhas x 10 colunas
product_variants: 1,009 linhas x 12 colunas
orders: 48,998 linhas x 13 colunas
order_items: 147,320 linhas x 8 colunas


## 3. Filtrar o produto e criar o dataset unificado

O produto `Bússola de Bordo 702` possui mais de um registro em `products.csv`. Portanto, todas as ocorrências são consideradas antes da busca pelas variantes.

In [4]:
TARGET_PRODUCT = 'Bússola de Bordo 702'
VALID_STATUSES = ['paid', 'confirmed']

# ---------------------------------------------------------
# 1. Localizar todas as ocorrências do produto
# ---------------------------------------------------------
target_products = products[
    products['name'].astype(str).str.strip().str.casefold()
    == TARGET_PRODUCT.casefold()
].copy()

if target_products.empty:
    raise ValueError(f'Produto não encontrado: {TARGET_PRODUCT}')

target_product_ids = target_products['id'].tolist()

print('Produtos encontrados:')
display(target_products[['id', 'name', 'is_active']])
print('product_ids considerados:', target_product_ids)

# ---------------------------------------------------------
# 2. Variantes relacionadas ao produto
# ---------------------------------------------------------
target_variants = variants[
    variants['product_id'].isin(target_product_ids)
][['id', 'product_id', 'sku', 'is_active']].copy()

target_variants = target_variants.rename(
    columns={'id': 'product_variant_id'}
)

if target_variants.empty:
    raise ValueError('Nenhuma variante encontrada para o produto.')

print('\nVariantes encontradas:')
display(target_variants)

# ---------------------------------------------------------
# 3. Filtrar order_items
# ---------------------------------------------------------
items = order_items[
    order_items['product_variant_id'].isin(
        target_variants['product_variant_id']
    )
][['order_id', 'product_variant_id', 'quantity']].copy()

# ---------------------------------------------------------
# 4. Preparar pedidos
# ---------------------------------------------------------
orders_selected = orders[
    ['id', 'placed_at', 'status']
].copy().rename(
    columns={
        'id': 'order_id',
        'placed_at': 'order_date',
    }
)

# ---------------------------------------------------------
# 5. Join itens -> pedidos
# ---------------------------------------------------------
unified_df = items.merge(
    orders_selected,
    on='order_id',
    how='inner',
    validate='many_to_one',
)

# ---------------------------------------------------------
# 6. Join itens -> variantes
# ---------------------------------------------------------
unified_df = unified_df.merge(
    target_variants[['product_variant_id', 'product_id']],
    on='product_variant_id',
    how='inner',
    validate='many_to_one',
)

# ---------------------------------------------------------
# 7. Adicionar nome do produto
# ---------------------------------------------------------
product_lookup = target_products[
    ['id', 'name']
].copy().rename(
    columns={
        'id': 'product_id',
        'name': 'product_name',
    }
)

unified_df = unified_df.merge(
    product_lookup,
    on='product_id',
    how='left',
    validate='many_to_one',
)

# ---------------------------------------------------------
# 8. Manter apenas pedidos que representam venda válida
# ---------------------------------------------------------
unified_df = unified_df[
    unified_df['status'].isin(VALID_STATUSES)
].copy()

# ---------------------------------------------------------
# 9. Tratamento de data e quantidade
# ---------------------------------------------------------
unified_df['order_date'] = pd.to_datetime(
    unified_df['order_date'],
    errors='coerce',
)

unified_df['quantity'] = pd.to_numeric(
    unified_df['quantity'],
    errors='coerce',
)

unified_df = unified_df.dropna(
    subset=['order_date', 'quantity']
).copy()

# ---------------------------------------------------------
# 10. Criar referência mensal
# ---------------------------------------------------------
unified_df['month'] = (
    unified_df['order_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# ---------------------------------------------------------
# 11. Dataset final
# ---------------------------------------------------------
unified_df = unified_df[
    [
        'order_date',
        'month',
        'product_id',
        'product_name',
        'product_variant_id',
        'order_id',
        'status',
        'quantity',
    ]
].sort_values('order_date').reset_index(drop=True)

print(f'\nLinhas no dataset unificado: {len(unified_df):,}')
print(f'Quantidade total vendida: {unified_df["quantity"].sum():,.0f}')
display(unified_df.head())

Produtos encontrados:


,id,name,is_active
73,74,Bússola de Bordo 702,True
239,240,Bússola de Bordo 702,True


product_ids considerados: [74, 240]

Variantes encontradas:


,product_variant_id,product_id,sku,is_active
146,147,74,LHN-677223,True
147,148,74,LHN-795790,True
485,486,240,LHN-304058,True



Linhas no dataset unificado: 408
Quantidade total vendida: 2,200


,order_date,month,product_id,product_name,product_variant_id,order_id,status,quantity
0,2020-01-05 21:45:00,2020-01-01,74,Bússola de Bordo 702,147,46233,paid,1
1,2020-01-11 11:20:28,2020-01-01,240,Bússola de Bordo 702,486,666,paid,9
2,2020-01-17 13:06:23,2020-01-01,74,Bússola de Bordo 702,148,21540,paid,9
3,2020-01-21 20:59:00,2020-01-01,74,Bússola de Bordo 702,147,11381,confirmed,4
4,2020-01-24 07:08:29,2020-01-01,74,Bússola de Bordo 702,147,3445,paid,6


## 4. Criar a série mensal de demanda

A quantidade vendida é somada por mês. Meses sem venda são preenchidos com zero para preservar a continuidade temporal.

In [5]:
monthly = (
    unified_df
    .groupby('month')['quantity']
    .sum()
    .sort_index()
    .rename('actual')
    .to_frame()
)

full_month_range = pd.date_range(
    start=monthly.index.min(),
    end=monthly.index.max(),
    freq='MS',
)

monthly = monthly.reindex(
    full_month_range,
    fill_value=0,
)

monthly.index.name = 'month'
monthly['actual'] = monthly['actual'].astype(float)

print('Período disponível:')
print(monthly.index.min().strftime('%Y-%m'), 'até', monthly.index.max().strftime('%Y-%m'))
display(monthly.tail(12))

Período disponível:
2020-01 até 2026-12


,actual
month,
2026-01-01,76.00
2026-02-01,55.00
2026-03-01,51.00
2026-04-01,23.00
2026-05-01,27.00
2026-06-01,31.00
2026-07-01,22.00
2026-08-01,25.00
2026-09-01,46.00


## 5. Separação treino e teste

- Treino: até dezembro de 2025.
- Teste: janeiro, fevereiro e março de 2026.

In [6]:
TRAIN_END = pd.Timestamp('2025-12-01')
TEST_START = pd.Timestamp('2026-01-01')
TEST_END = pd.Timestamp('2026-03-01')

train = monthly.loc[
    monthly.index <= TRAIN_END
].copy()

test = monthly.loc[
    (monthly.index >= TEST_START)
    & (monthly.index <= TEST_END)
].copy()

expected_test_months = pd.date_range(
    TEST_START,
    TEST_END,
    freq='MS',
)

if not test.index.equals(expected_test_months):
    raise ValueError(
        'O período de teste precisa conter exatamente jan, fev e mar/2026.'
    )

print(f'Treino: {train.index.min():%Y-%m} até {train.index.max():%Y-%m}')
print(f'Teste:  {test.index.min():%Y-%m} até {test.index.max():%Y-%m}')
display(test)

Treino: 2020-01 até 2025-12
Teste:  2026-01 até 2026-03


,actual
month,
2026-01-01,76.00
2026-02-01,55.00
2026-03-01,51.00


## 6. Baseline — média móvel dos últimos 3 meses

Para cada mês do teste, são utilizados somente os três meses anteriores ao mês previsto.

A avaliação abaixo é **walk-forward**: depois que janeiro termina, seu valor real pode ser utilizado para prever fevereiro; depois que fevereiro termina, seu valor real pode ser utilizado para prever março. Em nenhum momento o valor real do próprio mês previsto entra no cálculo da sua previsão.

In [7]:
def moving_average_forecast(series, forecast_months, window=3):
    predictions = []

    for forecast_month in forecast_months:
        previous_months = series.loc[
            series.index < forecast_month
        ].tail(window)

        if len(previous_months) < window:
            raise ValueError(
                f'Histórico insuficiente antes de {forecast_month:%Y-%m}.'
            )

        predictions.append({
            'month': forecast_month,
            'forecast': previous_months.mean(),
            'months_used': ', '.join(previous_months.index.strftime('%Y-%m')),
            'values_used': ', '.join(previous_months.astype(int).astype(str)),
        })

    return pd.DataFrame(predictions).set_index('month')


forecast = moving_average_forecast(
    monthly['actual'],
    expected_test_months,
    window=3,
)

forecast['actual'] = test['actual']
forecast['error'] = forecast['actual'] - forecast['forecast']
forecast['absolute_error'] = forecast['error'].abs()

display(forecast)

,forecast,months_used,values_used,actual,error,absolute_error
month,,,,,,
2026-01-01,32.67,"2025-10, 2025-11, 2025-12","25, 54, 19",76.00,43.33,43.33
2026-02-01,49.67,"2025-11, 2025-12, 2026-01","54, 19, 76",55.00,5.33,5.33
2026-03-01,50.00,"2025-12, 2026-01, 2026-02","19, 76, 55",51.00,1.00,1.00


## 7. Cálculo do MAE

In [8]:
mae = forecast['absolute_error'].mean()

print(f'MAE: {mae:.2f} unidades')

result = forecast[
    ['forecast', 'actual', 'absolute_error']
].copy()

result.columns = [
    'previsao_unidades',
    'real_unidades',
    'erro_absoluto',
]

result.index = result.index.strftime('%Y-%m')
display(result)

MAE: 16.56 unidades


,previsao_unidades,real_unidades,erro_absoluto
month,,,
2026-01,32.67,76.00,43.33
2026-02,49.67,55.00,5.33
2026-03,50.00,51.00,1.00


## 8. Validação dos valores esperados para os arquivos fornecidos

Com os CSVs disponibilizados no desafio e considerando apenas pedidos `paid` e `confirmed`, o resultado esperado é:

| Mês | Meses usados | Vendas anteriores | Previsão | Real | Erro absoluto |
|---|---|---|---:|---:|---:|
| Jan/2026 | Out-Nov-Dez/2025 | 25, 54, 19 | 32,67 | 76 | 43,33 |
| Fev/2026 | Nov-Dez/2025-Jan/2026 | 54, 19, 76 | 49,67 | 55 | 5,33 |
| Mar/2026 | Dez/2025-Jan-Fev/2026 | 19, 76, 55 | 50,00 | 51 | 1,00 |

**MAE esperado: 16,56 unidades.**

## 9. Respostas objetivas

### a. O baseline é adequado para esse produto?

O baseline é **adequado como referência inicial**, pois é simples, interpretável e respeita a ordem temporal dos dados. Entretanto, para este produto ele apresenta erro relevante no início do período de teste: em janeiro de 2026 a demanda real foi muito superior à média dos três meses anteriores. Com MAE de aproximadamente **16,56 unidades**, o método deve ser tratado como benchmark e não como solução definitiva de planejamento de estoque.

### b. Limitação do método

A média móvel de três meses reage apenas ao histórico recente e **não modela sazonalidade, tendência, promoções, preço, ruptura de estoque ou mudanças bruscas de demanda**. Por isso, pode atrasar a reação a picos ou quedas relevantes de vendas.